# 09 - Iterators, Generators, Decorators, Context Managers & Exceptions

Five features that share one idea: code that runs somewhere other than where you wrote it. An iterator suspends between items, a generator suspends mid-function, a decorator wraps a call you never see, a context manager owns the exits of a block, and an exception travels up frames until somebody claims it.

This is the hardest notebook in the set, and the gotchas are the point: a generator that is empty the second time you look, a `finally` that eats an exception whole, a decorator that renames every function in your traceback.

One convention throughout — the checker is strict on type, and a generator object is not a list. Whenever a question asks for a list, write the generator *and* a thin wrapper that returns `list(...)` of it.

---

**How to use this notebook**

1. Read the question. Write your answer in the code cell beneath it.
2. Run the cell. `check(...)` tells you PASS or FAIL, and shows the failing case.
3. Stuck? Open **Hint** first. Open **Solution** only after a genuine attempt.
4. Call `progress()` any time to see how far you have got.

**Difficulty labels** — questions run easiest to hardest down the notebook.

| | |
|---|---|
| L1 Extremely Easy | one concept, one line |
| L2 Easy | one concept applied, a small edge case |
| L3 Intermediate | 2-3 concepts combined, or a non-obvious edge case |
| L4 Somewhat Difficult | a design choice, correctness needs care |
| L5 Hard | multi-concept, subtle failure modes |


In [ ]:
# Run this cell first.
# Finds the repo root by walking up from wherever Jupyter was started, so this
# works whether you launched from the project root, from notebooks/, or
# installed the package with `pip install -e .`
import pathlib
import sys

if "sprintcheck" not in sys.modules:
    here = pathlib.Path.cwd().resolve()
    for folder in (here, *here.parents):
        if (folder / "sprintcheck" / "__init__.py").exists():
            sys.path.insert(0, str(folder))
            break

try:
    from sprintcheck import check, progress, reset
except ModuleNotFoundError:
    raise SystemExit(
        "Could not find the sprintcheck package.\n"
        "Open this notebook from inside the cloned repository, or run\n"
        "    pip install -e .\n"
        "from the project root and restart the kernel."
    )

print("Ready. Solve a question, then run check('Q-001', your_answer).")
print("Call progress() any time to see how far you have got.")


---

## L1 - Extremely Easy

### Q-315 - L1 Extremely Easy - Iterators

Write `first_two(values)` that returns the **tuple** of the first two items of any iterable.

Drive it by hand: get an iterator with `iter(values)` and pull from it with `next()` twice. No indexing, no slicing — the argument may be something that has no positions at all.

If there are fewer than two items, let `next` do what it already does.

<details><summary>Hint</summary>

`iter` turns a container into a one-shot cursor; `next` advances that cursor and hands you the item it passed. Call `iter` once and keep the result — a second `iter(values)` on a list would start over from the beginning.

</details>

<details><summary>Solution</summary>

```python
def first_two(values):
    it = iter(values)
    return (next(it), next(it))
```

**Why:** `for` is sugar for exactly this: call `iter`, call `next` until `StopIteration`. Holding the iterator in a variable is what makes the two `next` calls consecutive rather than both returning the first item, and it is the whole reason a generator can be consumed a piece at a time.

</details>

In [ ]:
def first_two(values):
    ...

check('Q-315', first_two)

### Q-316 - L1 Extremely Easy - Exceptions

Write `safe_div(a, b)` that returns `a / b`, or returns `None` when `b` is zero.

Catch the division by zero with `try`/`except` naming the exception explicitly — do not test `b` with an `if`, and do not write a bare `except:`.

<details><summary>Hint</summary>

Dividing by zero raises one specific exception whose name says exactly what happened. Name that class in the `except` clause; anything wider catches bugs you wanted to hear about.

</details>

<details><summary>Solution</summary>

```python
def safe_div(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        return None
```

**Why:** Python's culture is *ask forgiveness, not permission*: attempt the operation and handle the failure, rather than pre-checking every condition that could go wrong. The `if b == 0` version looks equivalent but drifts out of sync with reality the moment `b` is a type whose division fails for some other reason.

</details>

In [ ]:
def safe_div(a, b):
    ...

check('Q-316', safe_div)

---

## L2 - Easy

### Q-317 - L2 Easy - Generators

Write two things:

1. `gen_squares(n)` — a **generator function** that `yield`s `0*0`, `1*1`, ... up to but not including `n*n`.
2. `squares(n)` — returns `list(gen_squares(n))`, a plain **list of ints**.

`squares(4)` returns `[0, 1, 4, 9]`. The checker is strict on type, so a generator object will never pass where a list is expected — that wrapper is not decoration, it is the whole convention this notebook uses.

<details><summary>Hint</summary>

A function becomes a generator the moment its body contains `yield` — no special declaration, no `return` of anything. Loop over `range(n)` and yield once per step.

</details>

<details><summary>Solution</summary>

```python
def gen_squares(n):
    for i in range(n):
        yield i * i


def squares(n):
    return list(gen_squares(n))
```

**Why:** `yield` changes what the `def` *is*: calling `gen_squares(4)` runs no code at all and hands back a generator object. The classic surprise is printing that object and seeing `<generator object ...>` instead of numbers — you have the recipe, not the meal, until something iterates it.

</details>

In [ ]:
def gen_squares(n):
    ...


def squares(n):
    ...

check('Q-317', squares)

### Q-318 - L2 Easy - Generators

Write two things:

1. `naturals()` — an **infinite** generator yielding `0, 1, 2, 3, ...` forever, with a `while True` loop.
2. `first_naturals(n)` — returns a **list** of the first `n` values from a fresh `naturals()` generator.

`first_naturals(3)` returns `[0, 1, 2]`. Never call `list()` on the infinite generator itself.

<details><summary>Hint</summary>

An infinite generator is safe precisely because nothing runs until something asks. Take the prefix by calling `next` exactly `n` times — a comprehension over `range(n)` does it in one line.

</details>

<details><summary>Solution</summary>

```python
def naturals():
    n = 0
    while True:
        yield n
        n += 1


def first_naturals(n):
    gen = naturals()
    return [next(gen) for _ in range(n)]
```

**Why:** `while True` inside a generator is not a hang: each `yield` suspends the frame mid-loop and returns control to the caller, who decides whether there is ever a next iteration. That is what lets generators model streams, sensors, and `itertools.count` — sequences with no end that still cost O(1) memory.

</details>

In [ ]:
def naturals():
    ...


def first_naturals(n):
    ...

check('Q-318', first_naturals)

### Q-319 - L2 Easy - Exceptions

Write `attempt(text)` that records which blocks of a full `try`/`except`/`else`/`finally` actually run, and returns them as a **list of strings**.

Append `"try"` at the top of the `try` block, then call `int(text)`. Append `"except"` in an `except ValueError` block, `"else"` in the `else` block and `"finally"` in the `finally` block. Return the list.

`attempt("42")` returns `['try', 'else', 'finally']`.

<details><summary>Hint</summary>

`else` belongs to the *no exception was raised* path and `finally` to the *whatever happened* path. Keep the `try` block down to the one line that can actually fail; everything that depends on its success belongs in `else`.

</details>

<details><summary>Solution</summary>

```python
def attempt(text):
    events = []
    try:
        events.append("try")
        int(text)
    except ValueError:
        events.append("except")
    else:
        events.append("else")
    finally:
        events.append("finally")
    return events
```

**Why:** `else` exists so the `try` block can stay tiny: code moved into `else` still runs on success, but an exception it raises is no longer caught by your own `except`. Putting that code inside `try` instead is the classic way to accidentally swallow an unrelated `ValueError` from three frames down.

</details>

In [ ]:
def attempt(text):
    ...

check('Q-319', attempt)

### Q-320 - L2 Easy - Decorators

Write a decorator `double(func)` that returns a wrapper calling `func` and returning **twice** whatever it returned.

Then define `add(a, b)` returning `a + b`, decorated with `@double`, so that `add(1, 2)` returns `6`.

The wrapper must accept `*args, **kwargs` so it would work on any function, not only this one.

<details><summary>Hint</summary>

A decorator is just a function that takes a function and returns a replacement. `@double` above `def add` is exactly `add = double(add)` — the name ends up bound to your inner wrapper.

</details>

<details><summary>Solution</summary>

```python
def double(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) * 2
    return wrapper


@double
def add(a, b):
    return a + b
```

**Why:** There is no decorator machinery in the language beyond that rebinding — `@` is syntax for one function call. The inner `wrapper` keeps a reference to `func` through a closure, which is why the original is still reachable even though its name now points somewhere else.

</details>

In [ ]:
def double(func):
    ...


@double
def add(a, b):
    ...

check('Q-320', add)

### Q-321 - L2 Easy - Context Managers

Write a class `Tracker` that is a context manager over a shared list:

- `Tracker(events)` stores the list;
- `__enter__` appends `"enter"` and returns `self`;
- `__exit__` appends `"exit"` and returns `False`.

Then write `run_with(labels)` that makes a fresh `events = []`, opens `Tracker(events)` in a `with` statement, appends every string in `labels` inside the block, and returns `events`.

`run_with(["a"])` returns `['enter', 'a', 'exit']`.

<details><summary>Hint</summary>

`__exit__` takes three extra parameters — the exception type, value and traceback — even when nothing went wrong, in which case all three are `None`. Accept them and ignore them here.

</details>

<details><summary>Solution</summary>

```python
class Tracker:
    def __init__(self, events):
        self.events = events

    def __enter__(self):
        self.events.append("enter")
        return self

    def __exit__(self, exc_type, exc, tb):
        self.events.append("exit")
        return False


def run_with(labels):
    events = []
    with Tracker(events):
        for label in labels:
            events.append(label)
    return events
```

**Why:** The point of `with` is that `__exit__` runs on *every* way out of the block — falling off the end, `return`, `break`, or an exception — which is what makes it the right tool for anything that must be released. Whatever `__enter__` returns is what `as` binds, and returning `self` is only a convention, not a rule.

</details>

In [ ]:
class Tracker:
    ...


def run_with(labels):
    ...

check('Q-321', run_with)

### Q-322 - L2 Easy - Custom Exceptions

Define an exception class `NegativeError` that **subclasses `ValueError`**, then write `validate(n)`:

- return `n` unchanged when `n >= 0`;
- `raise NegativeError` with a message when `n` is negative.

Because `NegativeError` is a `ValueError`, existing callers that already write `except ValueError` keep working — the checker relies on that.

<details><summary>Hint</summary>

A custom exception needs no body at all: `class Name(Base): pass` is a complete definition. The decision that matters is which base class you inherit from, because that is what callers will be catching.

</details>

<details><summary>Solution</summary>

```python
class NegativeError(ValueError):
    pass


def validate(n):
    if n < 0:
        raise NegativeError(f"{n} is negative")
    return n
```

**Why:** Choosing the base class is the whole design: inherit from the exception a caller would *already* be catching, so adding your type breaks nobody, and callers who care can catch the narrow one. Subclassing `Exception` when `ValueError` was the honest answer forces every caller to learn your class name.

</details>

In [ ]:
class NegativeError(ValueError):
    ...


def validate(n):
    ...

check('Q-322', validate)

### Q-323 - L2 Easy - Iterators

Write `manual_sum(values)` that adds up an iterable of numbers and returns the total — **without a `for` loop** and without `sum()`.

Call `iter()` once, then loop with `while True` calling `next()`, and catch `StopIteration` to know when to stop and return.

An empty iterable gives `0`.

<details><summary>Hint</summary>

`StopIteration` is not an error condition — it is how an iterator says 'finished'. Catch it inside the `while` and return the running total from the handler.

</details>

<details><summary>Solution</summary>

```python
def manual_sum(values):
    it = iter(values)
    total = 0
    while True:
        try:
            total += next(it)
        except StopIteration:
            return total
```

**Why:** This is literally what `for` compiles to, which is why `StopIteration` is a signal rather than a failure. The practical consequence: a stray `StopIteration` raised *inside* a generator body used to silently end the generator — Python 3.7 fixed that by converting it to a `RuntimeError`, because the silent version hid real bugs.

</details>

In [ ]:
def manual_sum(values):
    ...

check('Q-323', manual_sum)

---

## L3 - Intermediate

### Q-324 - L3 Intermediate - Generators

Without running it, predict the output:

```python
gen = (n * n for n in range(3))
print(list(gen))
print(list(gen))
print(sum(gen))
```

Set `answer` to the three printed lines, e.g. `answer = "[0, 1, 4]\n[0, 1, 4]\n5"`.

<details><summary>Hint</summary>

A generator is a cursor, not a collection. Ask where that cursor is sitting after the first `list()` has finished with it.

</details>

<details><summary>Solution</summary>

```python
answer = "[0, 1, 4]\n[]\n0"
```

**Why:** A generator is exhausted after one pass and never rewinds — the second `list()` gets an immediate `StopIteration` and builds `[]`, with no error to warn you. This bites hardest when a generator is passed to two functions, or iterated once to count and again to use: store `list(gen)` if you need the values twice.

</details>

In [ ]:
answer = ...

check('Q-324', answer)

### Q-325 - L3 Intermediate - Generators

Without running it, predict the output:

```python
def noisy():
    print("start")
    yield 1
    print("middle")
    yield 2

g = noisy()
print("created")
print(next(g))
```

Set `answer` to the three printed lines, in the order they appear.

<details><summary>Hint</summary>

Work out how much of the generator body has executed at the moment `noisy()` returns. Then ask what the first `next()` runs, and where it stops.

</details>

<details><summary>Solution</summary>

```python
answer = "created\nstart\n1"
```

**Why:** Calling a generator function runs **none** of its body — it builds a suspended frame and returns immediately, so `"created"` prints first. The first `next()` runs up to and including the first `yield`, then freezes. Validation written at the top of a generator therefore fires late, or never: put argument checks in a plain wrapper function that returns the generator.

</details>

In [ ]:
answer = ...

check('Q-325', answer)

### Q-326 - L3 Intermediate - Exceptions

Without running it, predict the output:

```python
def f():
    try:
        return "try"
    finally:
        print("finally")

print(f())

def g():
    try:
        return "try"
    finally:
        return "finally"

print(g())
```

Set `answer` to the three printed lines, in order.

<details><summary>Hint</summary>

A `return` inside `try` does not leave the function immediately — the value is set aside first. Then ask what happens to that set-aside value if the `finally` block itself returns.

</details>

<details><summary>Solution</summary>

```python
answer = "finally\ntry\nfinally"
```

**Why:** `return` inside `try` evaluates its value, then runs `finally` before the function actually exits — so `f` prints `finally` *before* `print(f())` prints `try`. If `finally` issues its own `return`, it replaces the pending one outright: a `return` in `finally` also discards an in-flight exception, which is why linters flag it.

</details>

In [ ]:
answer = ...

check('Q-326', answer)

### Q-327 - L3 Intermediate - Decorators

Without running it, predict the output:

```python
import functools

def plain(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

def kept(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper

@plain
def alpha():
    pass

@kept
def beta():
    pass

print(alpha.__name__)
print(beta.__name__)
```

Set `answer` to the two printed lines.

<details><summary>Hint</summary>

After decoration the name `alpha` is bound to an object built inside `plain`. Ask what that object was called where it was defined.

</details>

<details><summary>Solution</summary>

```python
answer = "wrapper\nbeta"
```

**Why:** Without `functools.wraps`, the decorated name points at an object whose `__name__`, `__doc__` and `__module__` describe the wrapper — so help text, tracebacks, and anything that dispatches on `__name__` (pytest, Flask's route registry, `pickle`) all see `wrapper`. `wraps` copies that metadata across and records the original on `__wrapped__`; put it on every decorator you write.

</details>

In [ ]:
answer = ...

check('Q-327', answer)

### Q-328 - L3 Intermediate - Context Managers

Write a class `Swallow` whose `__exit__` returns `True`, and `shielded(error)`:

```python
def shielded(error):
    with Swallow():
        if error is not None:
            raise error
        return "body finished"
    return "suppressed"
```

`error` is either `None` or an exception **instance** to raise. Return the string the code above produces — write `Swallow` so that `shielded(ValueError("x"))` returns `'suppressed'` rather than blowing up.

<details><summary>Hint</summary>

`__exit__`'s return value is interpreted as a yes/no answer to one question: 'has this exception been dealt with?' Everything falsy — including the `None` you get by forgetting to return — means no.

</details>

<details><summary>Solution</summary>

```python
class Swallow:
    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        return True


def shielded(error):
    with Swallow():
        if error is not None:
            raise error
        return "body finished"
    return "suppressed"
```

**Why:** A truthy `__exit__` **suppresses** the exception and execution resumes on the line after the `with`, which is exactly how `contextlib.suppress` works. It is also the most dangerous thing a context manager can do by accident: `return True` written to mean 'cleanup succeeded' silently eats every error raised in the block.

</details>

In [ ]:
class Swallow:
    ...


def shielded(error):
    ...

check('Q-328', shielded)

### Q-329 - L3 Intermediate - Generators

Write `gen_flatten(groups)` — a generator that flattens one level using `yield from`, not a nested `for` — and `flatten_once(groups)` returning `list(gen_flatten(groups))`.

`flatten_once([[1, 2], [3]])` returns the **list** `[1, 2, 3]`. Any iterable may appear as a group, including a string.

<details><summary>Hint</summary>

`yield from some_iterable` yields every item of it in turn. That replaces the inner loop entirely, so the outer `for` is the only loop you need.

</details>

<details><summary>Solution</summary>

```python
def gen_flatten(groups):
    for group in groups:
        yield from group


def flatten_once(groups):
    return list(gen_flatten(groups))
```

**Why:** `yield from` is not only shorthand for `for x in it: yield x` — it also forwards `send`, `throw` and `close` to the sub-iterator and captures its return value, which is what makes generator delegation and recursion work (Q-349, Q-353). Note that a string is an iterable of characters, so it flattens into letters.

</details>

In [ ]:
def gen_flatten(groups):
    ...


def flatten_once(groups):
    ...

check('Q-329', flatten_once)

### Q-330 - L3 Intermediate - Decorators

Write a decorator `logged(func)` that wraps `func` and passes every argument straight through — but applies `functools.wraps` so the wrapper keeps the original's identity.

Decorate `triple(n)` (returning `n * 3`) with it, then write `describe(n)` returning the **tuple** `(triple.__name__, triple(n))`.

`describe(2)` returns `('triple', 6)`.

<details><summary>Hint</summary>

`functools.wraps(func)` is itself a decorator, applied to your inner wrapper function. It copies the metadata; it does not change what the wrapper does.

</details>

<details><summary>Solution</summary>

```python
import functools


def logged(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs)
    return wrapper


@logged
def triple(n):
    return n * 3


def describe(n):
    return (triple.__name__, triple(n))
```

**Why:** `wraps` copies `__name__`, `__doc__`, `__module__`, `__qualname__` and `__dict__`, and sets `__wrapped__` so `inspect.signature` can still find the real parameters. Skipping it makes every decorated function in a codebase report itself as `wrapper`, which turns tracebacks and log lines into a guessing game.

</details>

In [ ]:
import functools


def logged(func):
    ...


@logged
def triple(n):
    ...


def describe(n):
    ...

check('Q-330', describe)

### Q-331 - L3 Intermediate - Context Managers

Write `collecting(events)` as a context manager using the `@contextlib.contextmanager` decorator — **not** a class:

- append `"enter"`, then `yield events`, then append `"exit"`;
- the append of `"exit"` must be in a `finally` so it happens even if the block raises.

Then write `run_collecting(labels)` that makes a fresh list, opens `collecting(...) as log`, appends every string in `labels` to `log`, and returns the events list.

`run_collecting(["a"])` returns `['enter', 'a', 'exit']`.

<details><summary>Hint</summary>

The decorated generator must yield exactly once. Everything before the `yield` is `__enter__`, the yielded value is what `as` binds, and everything after is `__exit__`.

</details>

<details><summary>Solution</summary>

```python
from contextlib import contextmanager


@contextmanager
def collecting(events):
    events.append("enter")
    try:
        yield events
    finally:
        events.append("exit")


def run_collecting(labels):
    events = []
    with collecting(events) as log:
        for label in labels:
            log.append(label)
    return events
```

**Why:** `@contextmanager` turns a one-`yield` generator into a context manager by resuming it inside `__exit__`. The `try`/`finally` is mandatory, not stylistic: if the `with` body raises, that exception is thrown back in *at the `yield`*, so any cleanup written as a plain statement after the `yield` would simply never run.

</details>

In [ ]:
from contextlib import contextmanager


@contextmanager
def collecting(events):
    ...


def run_collecting(labels):
    ...

check('Q-331', run_collecting)

### Q-332 - L3 Intermediate - Exceptions

Write `parse(text)` that returns `int(text)`, but on `ValueError` raises `RuntimeError(f"bad input: {text!r}")` **from** the original exception.

Then write `inspect_parse(text)` returning a **tuple**:

- `("ok", value)` when the text parses;
- `(type(exc).__name__, type(exc.__cause__).__name__)` when it does not.

`inspect_parse("abc")` returns `('RuntimeError', 'ValueError')`.

<details><summary>Hint</summary>

`raise New(...) from original` sets one specific dunder attribute on the new exception. Bind the original with `except ValueError as exc` so you have something to chain from.

</details>

<details><summary>Solution</summary>

```python
def parse(text):
    try:
        return int(text)
    except ValueError as exc:
        raise RuntimeError(f"bad input: {text!r}") from exc


def inspect_parse(text):
    try:
        return ("ok", parse(text))
    except RuntimeError as exc:
        return (type(exc).__name__, type(exc.__cause__).__name__)
```

**Why:** `raise ... from exc` sets `__cause__`, which is what makes the traceback say *The above exception was the direct cause of the following exception* instead of the vaguer *During handling ... another exception occurred*. Translating a low-level error into a domain one without `from` throws away the original traceback's meaning — and `from None` deliberately hides it, which is occasionally what you want.

</details>

In [ ]:
def parse(text):
    ...


def inspect_parse(text):
    ...

check('Q-332', inspect_parse)

### Q-333 - L3 Intermediate - Iterators

Implement the iterator protocol by hand. Write a class `Countdown(start)` with:

- `__iter__` returning `self`;
- `__next__` yielding `start`, `start - 1`, ... down to `1`, then raising `StopIteration`.

Then write `countdown(start)` returning `list(Countdown(start))`. `countdown(3)` returns `[3, 2, 1]`; a `start` of `0` or less gives `[]`. No `yield` anywhere — that is Q-317's job.

<details><summary>Hint</summary>

The object has to remember where it is between calls, so the current position belongs on `self`. `__next__` takes no arguments and either returns the next item or raises.

</details>

<details><summary>Solution</summary>

```python
class Countdown:
    def __init__(self, start):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current <= 0:
            raise StopIteration
        self.current -= 1
        return self.current + 1


def countdown(start):
    return list(Countdown(start))
```

**Why:** `__iter__` and `__next__` are two different jobs: an *iterable* can hand out fresh cursors, an *iterator* is one cursor. Returning `self` from `__iter__` collapses them, which is why this object — like a generator — is single-use, and why a second `for` over the same `Countdown` yields nothing. Separate them (a `list` returns a new iterator each time) when you want repeat passes.

</details>

In [ ]:
class Countdown:
    ...


def countdown(start):
    ...

check('Q-333', countdown)

### Q-334 - L3 Intermediate - Custom Exceptions

Build a small exception hierarchy: `AppError(Exception)`, and two subclasses `NotFound(AppError)` and `Denied(AppError)`.

Write `handle(kind)`:

- `kind == "missing"` -> raise `NotFound`; `kind == "denied"` -> raise `Denied`; `kind == "other"` -> raise `ValueError`; anything else -> return `"ok"`;
- catch **only `AppError`** and return `f"app:{type(exc).__name__}"`.

`handle("missing")` returns `'app:NotFound'`. A `ValueError` must escape uncaught.

<details><summary>Hint</summary>

One `except` clause naming the base class catches every descendant, and the instance you bind still knows its own concrete class. Nothing outside the hierarchy should be listed.

</details>

<details><summary>Solution</summary>

```python
class AppError(Exception):
    pass


class NotFound(AppError):
    pass


class Denied(AppError):
    pass


def handle(kind):
    try:
        if kind == "missing":
            raise NotFound("missing")
        if kind == "denied":
            raise Denied("denied")
        if kind == "other":
            raise ValueError("other")
        return "ok"
    except AppError as exc:
        return f"app:{type(exc).__name__}"
```

**Why:** A single base class per library is the standard shape: callers who want everything write `except AppError`, callers who want one case write `except NotFound`, and you can add subclasses later without breaking either. The escaping `ValueError` is the feature — an error outside your domain should not be quietly relabelled as one of yours.

</details>

In [ ]:
class AppError(Exception):
    ...


def handle(kind):
    ...

check('Q-334', handle)

### Q-335 - L3 Intermediate - Generators

Write `gen_take(iterable, n)` — a generator yielding at most the first `n` items of `iterable` — and `take(iterable, n)` returning `list(gen_take(iterable, n))`.

It must stop early: pulling `n` items from a million-item source may touch only `n` of them, and asking for more items than exist returns what there is rather than raising.

`take([1, 2, 3, 4], 2)` returns the **list** `[1, 2]`.

<details><summary>Hint</summary>

Drive the source with an explicit iterator so you control how many items are pulled. Running out early is signalled by `StopIteration` from `next` — catch it and `return` from the generator.

</details>

<details><summary>Solution</summary>

```python
def gen_take(iterable, n):
    it = iter(iterable)
    for _ in range(n):
        try:
            yield next(it)
        except StopIteration:
            return


def take(iterable, n):
    return list(gen_take(iterable, n))
```

**Why:** A bare `return` inside a generator ends it — it does not return a value to the `for` loop, it raises `StopIteration` behind the scenes. Catching the inner `StopIteration` explicitly is required since Python 3.7: letting it escape the generator body now becomes a `RuntimeError` instead of quietly ending the stream. `itertools.islice` is the production version of this.

</details>

In [ ]:
def gen_take(iterable, n):
    ...


def take(iterable, n):
    ...

check('Q-335', take)

### Q-336 - L3 Intermediate - Generators

Write `running_total(numbers)` returning the **list** of running totals: `running_total([1, 2, 3])` returns `[1, 3, 6]`, and an empty input gives `[]`.

The work must be done by a **generator defined inside** `running_total` — a nested `def` that `yield`s the total after each number — with the outer function returning `list(...)` of it.

Your solution must contain `yield`: building the list directly does not count, even though it would produce the same answer.

<details><summary>Hint</summary>

Carry the total in a local variable across `yield`s — a generator's locals survive suspension, which is exactly what makes this shape work without a class or a global. The nested generator closes over `numbers`, so it needs no parameter of its own.

</details>

<details><summary>Solution</summary>

```python
def running_total(numbers):
    def gen():
        total = 0
        for n in numbers:
            total += n
            yield total

    return list(gen())
```

**Why:** The generator frame keeps its local variables alive between `yield`s, so accumulating state needs no object and no `nonlocal`. That is why generators are the natural fit for streaming aggregates — `itertools.accumulate` is this function, and it works on a feed that never ends.

</details>

In [ ]:
def running_total(numbers):
    ...

check('Q-336', running_total)

---

## L4 - Somewhat Difficult

### Q-337 - L4 Somewhat Difficult - Decorators

Write a **decorator factory** `repeat(times)`: `@repeat(3)` on a function makes every call run it three times and return a **list** of the three results.

Use `functools.wraps` on the wrapper. Then define `shout(word)` returning `word.upper()`, decorated with `@repeat(3)`, so `shout("hi")` returns `['HI', 'HI', 'HI']`.

<details><summary>Hint</summary>

`@repeat(3)` is a *call* first, then a decoration — `shout = repeat(3)(shout)`. That means one more level of nesting than Q-320: three functions, not two.

</details>

<details><summary>Solution</summary>

```python
import functools


def repeat(times):
    def decorate(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            return [func(*args, **kwargs) for _ in range(times)]
        return wrapper
    return decorate


@repeat(3)
def shout(word):
    return word.upper()
```

**Why:** A decorator with arguments is not a decorator — it is a function that *returns* one, which is why the body needs three nested `def`s. Forgetting the middle layer gives the classic `TypeError: decorate() missing 1 required positional argument`, or worse, a decorator that silently receives your argument where it expected the function.

</details>

In [ ]:
import functools


def repeat(times):
    ...


# once repeat works, decorate shout with @repeat(3)
def shout(word):
    ...

check('Q-337', shout)

### Q-338 - L4 Somewhat Difficult - Decorators

Write two decorators: `add_exclaim` appends `"!"` to the wrapped function's string result, `add_question` appends `"?"`.

Stack **both** on `say(word)` (which returns `word`) so that `say("hi")` returns `'hi?!'` — the `?` applied first, then the `!`.

The order of the two `@` lines is the entire question.

<details><summary>Hint</summary>

Decorators apply bottom-up: the one closest to the `def` wraps the raw function, and the one above wraps that. So the result you see is built outermost-last.

</details>

<details><summary>Solution</summary>

```python
def add_exclaim(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + "!"
    return wrapper


def add_question(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs) + "?"
    return wrapper


@add_exclaim
@add_question
def say(word):
    return word
```

**Why:** Stacked decorators read like nested calls written from the inside out: `say = add_exclaim(add_question(say))`. Application order is bottom-up, but *execution* order at call time is top-down — the outermost wrapper runs first and the original runs last, which is why `@app.route` must sit above `@login_required` and not below it.

</details>

In [ ]:
def add_exclaim(func):
    ...


def add_question(func):
    ...


def say(word):
    return word

check('Q-338', say)

### Q-339 - L4 Somewhat Difficult - Decorators

Write a decorator `memoize(func)` that caches results in a dict keyed by the single argument, so a repeated argument never reaches `func` again.

Then write `measure(inputs)` that — **inside the function**, so each call starts with an empty cache — defines a memoized `square(n)` recording each real call, maps it over `inputs`, and returns the tuple `(results_list, number_of_real_calls)`.

`measure([2, 2, 3])` returns `([4, 4, 9], 2)`.

<details><summary>Hint</summary>

The cache dict lives in the decorator's scope, not inside the wrapper — created once per decoration, then closed over. Defining the decorated function inside `measure` is what keeps the cache from leaking between calls.

</details>

<details><summary>Solution</summary>

```python
import functools


def memoize(func):
    cache = {}

    @functools.wraps(func)
    def wrapper(n):
        if n not in cache:
            cache[n] = func(n)
        return cache[n]

    return wrapper


def measure(inputs):
    calls = []

    @memoize
    def square(n):
        calls.append(n)
        return n * n

    results = [square(n) for n in inputs]
    return (results, len(calls))
```

**Why:** The cache is per-decoration, so a module-level memoized function keeps its entries for the life of the process — which is a leak if arguments are unbounded, and a correctness bug if `func` is not pure. `n not in cache` rather than `cache.get(n)` matters too: a legitimately cached `None` or `0` would defeat the truthiness check. `functools.lru_cache` is the real one.

</details>

In [ ]:
import functools


def memoize(func):
    ...


def measure(inputs):
    ...

check('Q-339', measure)

### Q-340 - L4 Somewhat Difficult - Generators

Prove laziness by observation. Write `counted(numbers, log)` — a generator that, for each number, appends it to `log` and then yields it doubled.

Then write `observe(numbers)` returning the **3-tuple** `(items_logged_before_any_next, first_value, items_logged_after_one_next)`:

create the generator, record `len(log)`, call `next()` once, and record `len(log)` again.

`observe([5, 6, 7])` returns `(0, 10, 1)`. If the source is empty, let the `next()` raise.

<details><summary>Hint</summary>

Nothing in the generator body has run at the moment the generator object is created. One `next()` runs exactly up to the first `yield` and no further.

</details>

<details><summary>Solution</summary>

```python
def counted(numbers, log):
    for n in numbers:
        log.append(n)
        yield n * 2


def observe(numbers):
    log = []
    gen = counted(numbers, log)
    created = len(log)
    first = next(gen)
    return (created, first, len(log))
```

**Why:** A generator does exactly as much work as it is asked for and no more — one `next()` touches one source item, which is why a generator pipeline over a 10 GB file never loads more than one record. The flip side is that side effects you write in a generator happen *when consumed*, not when called, so a generator that is built and never iterated does nothing at all.

</details>

In [ ]:
def counted(numbers, log):
    ...


def observe(numbers):
    ...

check('Q-340', observe)

### Q-341 - L4 Somewhat Difficult - Generators

Build a two-stage generator pipeline:

- `evens(numbers)` — a generator yielding only the even numbers;
- `scaled(numbers, factor)` — a generator yielding each number times `factor`;
- `pipeline(numbers, factor)` — returns `list(scaled(evens(numbers), factor))`, a plain **list**.

`pipeline([1, 2, 3, 4], 10)` returns `[20, 40]`. Neither generator may build an intermediate list.

<details><summary>Hint</summary>

Each stage takes an iterable and yields an iterable, so stages compose by being passed to one another. No stage ever needs to know how long the stream is.

</details>

<details><summary>Solution</summary>

```python
def evens(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n


def scaled(numbers, factor):
    for n in numbers:
        yield n * factor


def pipeline(numbers, factor):
    return list(scaled(evens(numbers), factor))
```

**Why:** Chained generators form a pull-based pipeline: the outer `list()` asks `scaled` for one item, which asks `evens` for one item, which asks the source — so peak memory is one item per stage regardless of input size. The equivalent list-at-each-stage version materialises the whole dataset three times, which is the difference between streaming a log file and running out of RAM.

</details>

In [ ]:
def evens(numbers):
    ...


def scaled(numbers, factor):
    ...


def pipeline(numbers, factor):
    ...

check('Q-341', pipeline)

### Q-342 - L4 Somewhat Difficult - Context Managers

Write a class `Report` whose `__exit__` **inspects** the exception it is given and records it on `self.seen`:

- `None` when the block finished cleanly;
- the tuple `(exc_type.__name__, str(exc))` otherwise.

`__exit__` must return `False` so the exception still propagates.

Then write `watch(error)`: build a `Report`, run a `with` block that raises `error` when it is not `None` (catching it outside the `with`), and return `reporter.seen`.

<details><summary>Hint</summary>

`__exit__(self, exc_type, exc, tb)` receives the class, the instance and the traceback — all three are `None` on the clean path, which is the test you need.

</details>

<details><summary>Solution</summary>

```python
class Report:
    def __init__(self):
        self.seen = None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.seen = None if exc_type is None else (exc_type.__name__, str(exc))
        return False


def watch(error):
    reporter = Report()
    try:
        with reporter:
            if error is not None:
                raise error
    except Exception:
        pass
    return reporter.seen
```

**Why:** Observing without suppressing is the useful default: return `False` (or nothing at all) and you get a hook that can log, time, or roll back while leaving error handling to the caller. Note `str(KeyError("k"))` is `"'k'"` with quotes — `KeyError` repr's its argument, which is why its messages look odd in logs.

</details>

In [ ]:
class Report:
    ...


def watch(error):
    ...

check('Q-342', watch)

### Q-343 - L4 Somewhat Difficult - Context Managers

Write `guarded(events)` with `@contextlib.contextmanager`:

- append `"open"`, then `yield`;
- catch `ValueError` around the `yield` and append `"handled"` (swallowing it);
- append `"close"` in a `finally`.

Then write `run_guarded(error)`: fresh list, `with guarded(events)`, append `"body"`, raise `error` when it is not `None`, return the list.

A `ValueError` gives `['open', 'body', 'handled', 'close']`; any other exception propagates out of `run_guarded` untouched.

<details><summary>Hint</summary>

An exception raised in the `with` body arrives at the `yield` inside the generator, so you catch it with an ordinary `try`/`except` wrapped around that `yield`. Catching it is what suppresses it.

</details>

<details><summary>Solution</summary>

```python
from contextlib import contextmanager


@contextmanager
def guarded(events):
    events.append("open")
    try:
        yield
    except ValueError:
        events.append("handled")
    finally:
        events.append("close")


def run_guarded(error):
    events = []
    with guarded(events):
        events.append("body")
        if error is not None:
            raise error
    return events
```

**Why:** In a `@contextmanager` generator, catching the exception at the `yield` is the equivalent of `__exit__` returning `True`, and letting it through is the equivalent of returning `False` — no boolean bookkeeping. Watch the return path though: when the body raises and the manager swallows it, the `with` block never finishes, so `run_guarded` returns from *after* the block.

</details>

In [ ]:
from contextlib import contextmanager


@contextmanager
def guarded(events):
    ...


def run_guarded(error):
    ...

check('Q-343', run_guarded)

### Q-344 - L4 Somewhat Difficult - Exceptions

Write `risky(n, log)` that returns `10 / n`, but on `ZeroDivisionError` appends `"logged"` to `log` and then **re-raises the same exception with a bare `raise`** — not `raise ZeroDivisionError(...)`.

Then write `report(n)` returning the tuple `(log, value)` on success and `(log, None)` when the division failed.

`report(0)` returns `(['logged'], None)`; `report(5)` returns `([], 2.0)`.

<details><summary>Hint</summary>

Inside an `except` block, `raise` on its own re-raises the exception currently being handled. That is different from raising a fresh instance of the same class in one specific, important way.

</details>

<details><summary>Solution</summary>

```python
def risky(n, log):
    try:
        return 10 / n
    except ZeroDivisionError:
        log.append("logged")
        raise


def report(n):
    log = []
    try:
        value = risky(n, log)
    except ZeroDivisionError:
        return (log, None)
    return (log, value)
```

**Why:** A bare `raise` preserves the original exception object **and its traceback**, so the report still points at the line that actually failed; `raise ZeroDivisionError(...)` throws that away and blames your handler. Log-and-re-raise is the right shape for a middle layer: record the context, then let the caller decide policy.

</details>

In [ ]:
def risky(n, log):
    ...


def report(n):
    ...

check('Q-344', report)

### Q-345 - L4 Somewhat Difficult - Exceptions

Without running it, predict the output:

```python
def boom():
    raise KeyboardInterrupt

def run(job):
    try:
        job()
    except:
        return "handled"
    return "fine"

print(run(boom))

def run_narrow(job):
    try:
        job()
    except Exception:
        return "handled"
    return "fine"

try:
    print(run_narrow(boom))
except KeyboardInterrupt:
    print("escaped")
```

Set `answer` to the two printed lines.

<details><summary>Hint</summary>

`KeyboardInterrupt` does not sit where you might assume in the exception hierarchy. Find its base class, then ask which of the two `except` clauses can possibly match it.

</details>

<details><summary>Solution</summary>

```python
answer = "handled\nescaped"
```

**Why:** `KeyboardInterrupt`, `SystemExit` and `GeneratorExit` inherit from `BaseException`, **not** `Exception`, precisely so that `except Exception` leaves them alone. A bare `except:` catches `BaseException` and therefore makes Ctrl-C do nothing — the reason bare `except` is the single most reliable smell in a Python review.

</details>

In [ ]:
answer = ...

check('Q-345', answer)

### Q-346 - L4 Somewhat Difficult - Exceptions

Write `classify(error)` that takes an exception **instance**, raises it inside a `try`, and returns a label from a chain of `except` clauses:

- `ZeroDivisionError` -> `'divide by zero'`
- `ArithmeticError` -> `'arithmetic'`
- `LookupError` -> `'lookup'`
- anything else -> `'other'`

`ZeroDivisionError` is a subclass of `ArithmeticError`, and both `KeyError` and `IndexError` are subclasses of `LookupError` — so the order of your clauses decides whether the first rule is reachable at all.

<details><summary>Hint</summary>

`except` clauses are tested top to bottom and the first match wins; a base class listed early shadows every subclass listed later. Sort them most-specific-first.

</details>

<details><summary>Solution</summary>

```python
def classify(error):
    try:
        raise error
    except ZeroDivisionError:
        return "divide by zero"
    except ArithmeticError:
        return "arithmetic"
    except LookupError:
        return "lookup"
    except Exception:
        return "other"
```

**Why:** Putting `except ArithmeticError` above `except ZeroDivisionError` makes the second clause dead code — and Python will not warn you, because it is perfectly legal. This is the same most-specific-first rule as an `isinstance` chain, and it is why `except Exception` belongs last or not at all.

</details>

In [ ]:
def classify(error):
    ...

check('Q-346', classify)

### Q-347 - L4 Somewhat Difficult - Iterators

`iter()` has a two-argument form: `iter(callable, sentinel)` calls the zero-argument callable over and over and stops — without yielding it — the moment the result equals `sentinel`.

Write `drain(items, sentinel)`: copy `items` into a local list, define a zero-argument `pull()` that pops from the front (returning `sentinel` when empty), and return `list(iter(pull, sentinel))`.

`drain([1, 2, 0, 3], 0)` returns the **list** `[1, 2]` — everything before the sentinel, and nothing after.

<details><summary>Hint</summary>

The callable takes no arguments, so whatever state it reads must come from the enclosing scope. Returning the sentinel when the source runs out is what terminates the loop for you.

</details>

<details><summary>Solution</summary>

```python
def drain(items, sentinel):
    source = list(items)

    def pull():
        return source.pop(0) if source else sentinel

    return list(iter(pull, sentinel))
```

**Why:** This form exists for the read-until-marker pattern: `iter(lambda: f.read(4096), b"")` streams a file in blocks, and `iter(sock.recv_line, "")` drains a socket — both without a `while True` and a `break`. The comparison is `==`, not `is`, so a sentinel that compares equal to a legitimate value will truncate your stream early.

</details>

In [ ]:
def drain(items, sentinel):
    ...

check('Q-347', drain)

### Q-348 - L4 Somewhat Difficult - Custom Exceptions

Write `ValidationError(ValueError)` carrying **structured data**, not just a message: `ValidationError(field, message)` calls `super().__init__(f"{field}: {message}")` and stores `self.field` and `self.message`.

Write `validate_name(record)`: raise `ValidationError("name", "missing")` when `record` has no `"name"` key, `ValidationError("name", "empty")` when the value is empty, otherwise return the name.

Write `describe_record(record)` returning the tuple `("ok", name)` on success, or `(exc.field, exc.message)` on failure.

<details><summary>Hint</summary>

Call `super().__init__` with the human-readable message so `str(exc)` still works, then set your own attributes. A handler should never have to parse a message string to learn which field failed.

</details>

<details><summary>Solution</summary>

```python
class ValidationError(ValueError):
    def __init__(self, field, message):
        super().__init__(f"{field}: {message}")
        self.field = field
        self.message = message


def validate_name(record):
    if "name" not in record:
        raise ValidationError("name", "missing")
    if not record["name"]:
        raise ValidationError("name", "empty")
    return record["name"]


def describe_record(record):
    try:
        return ("ok", validate_name(record))
    except ValidationError as exc:
        return (exc.field, exc.message)
```

**Why:** Exceptions are ordinary objects, so attach whatever the handler needs to decide — field names, retry-after seconds, the offending row — instead of forcing it to regex your message. Forgetting `super().__init__` is the usual slip: the attributes work, but `str(exc)` comes out empty and the traceback tells you nothing.

</details>

In [ ]:
class ValidationError(ValueError):
    ...


def validate_name(record):
    ...


def describe_record(record):
    ...

check('Q-348', describe_record)

### Q-349 - L4 Somewhat Difficult - Generators

Write `deep_flatten(values)` returning a flat **list** of every non-list item, at any depth: `deep_flatten([1, [2, [3, 4]], 5])` returns `[1, 2, 3, 4, 5]`. Only `list` counts as nesting — strings stay whole.

Do it with a **generator nested inside** `deep_flatten` that walks the items and, on hitting a sub-list, delegates with `yield from deep_flatten(item)` — so the recursion runs through the outer function. The outer function returns `list(...)` of that generator.

Your solution must contain `yield` and `deep_flatten` must call itself.

<details><summary>Hint</summary>

For each item: if it is a `list`, hand the whole sub-stream over to a recursive call; otherwise yield the item. `isinstance` decides which, and `yield from` accepts a list just as happily as a generator.

</details>

<details><summary>Solution</summary>

```python
def deep_flatten(values):
    def gen(items):
        for item in items:
            if isinstance(item, list):
                yield from deep_flatten(item)
            else:
                yield item

    return list(gen(values))
```

**Why:** This is the reason `yield from` was added: without it the recursive case needs `for x in deep_flatten(item): yield x`, and every level of nesting adds another Python-level loop to every item's journey to the top. The `isinstance` guard has to be there because a string is iterable — recursing into `"ab"` would yield `"a"`, which is itself a one-character string, and the recursion never bottoms out.

</details>

In [ ]:
def deep_flatten(values):
    ...

check('Q-349', deep_flatten)

### Q-350 - L4 Somewhat Difficult - Context Managers

Write a class `Session(log)` modelling a resource that is only usable while open:

- `__enter__` sets `self.open = True`, appends `"open"` to the log, returns `self`;
- `write(text)` raises `RuntimeError` if not open, else appends `text`;
- `__exit__` sets `self.open = False`, appends `"close"`, returns `False`.

Then write `transcript(messages)` that creates a fresh log and a `Session`, and writes every message **inside a `with` statement**, returning the log. `transcript(["a"])` returns `['open', 'a', 'close']`.

Your solution must use a `with` statement — calling `__enter__` and `__exit__` by hand does not count.

<details><summary>Hint</summary>

The `open` flag is the whole point of the design: it makes misuse outside the block raise loudly instead of half-working. `__exit__` clears it whichever way the block ends.

</details>

<details><summary>Solution</summary>

```python
class Session:
    def __init__(self, log):
        self.log = log
        self.open = False

    def __enter__(self):
        self.open = True
        self.log.append("open")
        return self

    def write(self, text):
        if not self.open:
            raise RuntimeError("session is closed")
        self.log.append(text)

    def __exit__(self, exc_type, exc, tb):
        self.open = False
        self.log.append("close")
        return False


def transcript(messages):
    log = []
    session = Session(log)
    with session:
        for message in messages:
            session.write(message)
    return log
```

**Why:** Guarding operations on an `open` flag is what turns a context manager from a convenience into a contract — the object refuses to be used at the wrong time, rather than silently writing to a closed handle. This is exactly why `open()` files raise `ValueError: I/O operation on closed file`, and why `with` is preferred over remembering to call `.close()`.

</details>

In [ ]:
class Session:
    ...


def transcript(messages):
    ...

check('Q-350', transcript)

### Q-351 - L4 Somewhat Difficult - Exceptions

Without running it, predict the output:

```python
def f():
    try:
        raise ValueError("boom")
    finally:
        return "swallowed"

print(f())

def g():
    try:
        raise ValueError("boom")
    finally:
        print("cleanup")

try:
    g()
except ValueError as exc:
    print(f"caught {exc}")
```

Set `answer` to the three printed lines, in order. Neither function has an `except` clause — think about what a `return` inside `finally` does to an exception that is already on its way out.

<details><summary>Hint</summary>

`finally` runs while the exception is in flight, before it leaves the function. Ask what is left of that in-flight exception once `finally` performs a normal exit of its own.

</details>

<details><summary>Solution</summary>

```python
answer = "swallowed\ncleanup\ncaught boom"
```

**Why:** A `return` (or `break`, or `continue`) inside `finally` **discards the in-flight exception entirely** — `f` reports success and the `ValueError` is gone without a trace, no log, no traceback. `g` shows the normal behaviour: `finally` runs its cleanup and the exception carries on. Never put control flow in a `finally` block.

</details>

In [ ]:
answer = ...

check('Q-351', answer)

---

## L5 - Hard

### Q-352 - L5 Hard - Generators

Generators receive as well as produce. Write `accumulator()` — an infinite generator whose `yield` expression is assigned: `value = yield total`. It keeps a running `total`, treats a received `None` as `0`, and yields the total each time.

Then write `feed(numbers)`: create the generator, **prime it** with one `next()`, then `send()` each number and collect the values it yields back into a **list**.

`feed([1, 2, 3])` returns `[1, 3, 6]`.

<details><summary>Hint</summary>

`gen.send(x)` resumes the generator, makes the paused `yield` expression evaluate to `x`, and returns the next yielded value. A brand-new generator is not paused at a `yield` yet, which is why the first call has to be `next()`.

</details>

<details><summary>Solution</summary>

```python
def accumulator():
    total = 0
    while True:
        value = yield total
        if value is None:
            value = 0
        total += value


def feed(numbers):
    gen = accumulator()
    next(gen)
    return [gen.send(n) for n in numbers]
```

**Why:** `yield` is an expression, not a statement: it produces whatever `send` passed in, which turns a generator into a two-way coroutine. Priming is mandatory — `send(1)` on a fresh generator raises `TypeError: can't send non-None value to a just-started generator` — and `next(gen)` is exactly `gen.send(None)`, which is also why the `None` guard is needed.

</details>

In [ ]:
def accumulator():
    ...


def feed(numbers):
    ...

check('Q-352', feed)

### Q-353 - L5 Hard - Generators

A generator can `return` a value, and `yield from` captures it.

- `counter(values)` — yields each value, then `return`s how many it yielded.
- `wrapper(groups)` — for each group, does `n = yield from counter(group)` and collects the `n`s; at the end `return`s that list of counts.
- `collect(groups)` — drives `wrapper` with `next()` in a loop, gathering the yielded items, and catches `StopIteration` to read `stop.value`. Return the **tuple** `(items_list, counts_list)`.

`collect([[1, 2], [3]])` returns `([1, 2, 3], [2, 1])`.

<details><summary>Hint</summary>

A generator's `return` value is not yielded — it is attached to the `StopIteration` that ends it, as `.value`. `yield from` unwraps that for you; at the outermost level you have to catch it yourself.

</details>

<details><summary>Solution</summary>

```python
def counter(values):
    count = 0
    for value in values:
        yield value
        count += 1
    return count


def wrapper(groups):
    totals = []
    for group in groups:
        n = yield from counter(group)
        totals.append(n)
    return totals


def collect(groups):
    gen = wrapper(groups)
    items = []
    while True:
        try:
            items.append(next(gen))
        except StopIteration as stop:
            return (items, stop.value)
```

**Why:** `return x` in a generator is sugar for `raise StopIteration(x)`, which is why a `for` loop discards it completely — loops only see yielded values. `yield from` is the only construct that reads it, and that channel is what made generator-based coroutines possible before `async`/`await` existed.

</details>

In [ ]:
def counter(values):
    ...


def wrapper(groups):
    ...


def collect(groups):
    ...

check('Q-353', collect)

### Q-354 - L5 Hard - Iterators

Write `windows(iterable, size)` returning a **list of tuples**: every consecutive run of `size` items. `windows([1, 2, 3, 4], 2)` returns `[(1, 2), (2, 3), (3, 4)]`; a source shorter than `size` gives `[]`. `size` is always at least 1.

The sliding itself must be done by a **generator nested inside** `windows`, with the outer function returning `list(...)` of it.

It must work on a one-pass iterator: no indexing, no slicing, and no `zip` over shifted copies. Use a `collections.deque` with `maxlen`.

<details><summary>Hint</summary>

A `deque(maxlen=size)` drops from the left automatically when it is full, so appending is the whole sliding step. Yield a *copy* — `tuple(window)` — because the deque keeps mutating underneath you.

</details>

<details><summary>Solution</summary>

```python
from collections import deque


def windows(iterable, size):
    def gen():
        window = deque(maxlen=size)
        for item in iterable:
            window.append(item)
            if len(window) == size:
                yield tuple(window)

    return list(gen())
```

**Why:** The `maxlen` deque gives you O(1) sliding with bounded memory — the whole point, since the source may be a stream with no length. Yielding `window` itself instead of `tuple(window)` is the bug that makes every result in the output list look identical: they would all be the same mutating object. `itertools.pairwise` is the `size=2` case in the standard library.

</details>

In [ ]:
from collections import deque


def windows(iterable, size):
    ...

check('Q-354', windows)

### Q-355 - L5 Hard - Context Managers

Reimplement `contextlib.suppress`. Write a class `only(*exc_types)` that suppresses an exception from its block when it is an instance of any listed type, records `type(exc).__name__` on `self.caught`, and lets everything else propagate.

Then write `guard(error, types)`:

```python
box = only(*types)
with box:
    if error is not None:
        raise error
    return ("ran", None)
return ("suppressed", box.caught)
```

`guard(ValueError("x"), (ValueError,))` returns `('suppressed', 'ValueError')`. `only()` with no types suppresses nothing. Do not import `contextlib.suppress`.

<details><summary>Hint</summary>

`__exit__` gets the exception *class* as its first argument, so the membership test is `issubclass`, not `isinstance` — and it has to short-circuit on the clean path where that argument is `None`.

</details>

<details><summary>Solution</summary>

```python
class only:
    def __init__(self, *exc_types):
        self.exc_types = exc_types
        self.caught = None

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        if exc_type is not None and issubclass(exc_type, self.exc_types):
            self.caught = exc_type.__name__
            return True
        return False


def guard(error, types):
    box = only(*types)
    with box:
        if error is not None:
            raise error
        return ("ran", None)
    return ("suppressed", box.caught)
```

**Why:** `issubclass` against a tuple handles the whole hierarchy for free, so `only(ArithmeticError)` catches a `ZeroDivisionError` exactly as an `except` clause would — and an empty tuple is always `False`, which is why `only()` correctly suppresses nothing. The reason `suppress` exists at all is that `try: ... except X: pass` is three lines whose `pass` reviewers can never tell from an accident.

</details>

In [ ]:
class only:
    ...


def guard(error, types):
    ...

check('Q-355', guard)

### Q-356 - L5 Hard - Custom Exceptions

Write `chain_messages(exc)` that walks an exception chain and returns the **list of `str(...)` messages**, outermost first. At each step follow `__cause__` if it is set, otherwise `__context__`, stopping at `None`.

Supporting code: `parse(text)` raises `ValueError(f"not a number: {text}")` unless `text.isdigit()`; `load(text, chained)` catches that and raises `StepError("parse step failed")` — **`from exc` when `chained` is true, and a plain `raise` when it is false**. `StepError` is your own `Exception` subclass.

Write `trace(text, chained=True)` returning `("ok", value)` on success, or `("failed", chain_messages(exc))` when `StepError` escapes.

`trace("abc")` returns `('failed', ['parse step failed', 'not a number: abc'])` — and so does `trace("abc", chained=False)`.

<details><summary>Hint</summary>

Two different dunder attributes record a chain: one you set deliberately with `from`, one Python fills in automatically whenever you raise inside an `except` block. Prefer the deliberate one when both are present.

</details>

<details><summary>Solution</summary>

```python
class StepError(Exception):
    pass


def chain_messages(exc):
    messages = []
    current = exc
    while current is not None:
        messages.append(str(current))
        if current.__cause__ is not None:
            current = current.__cause__
        else:
            current = current.__context__
    return messages


def parse(text):
    if not text.isdigit():
        raise ValueError(f"not a number: {text}")
    return int(text)


def load(text, chained):
    try:
        return parse(text)
    except ValueError as exc:
        if chained:
            raise StepError("parse step failed") from exc
        raise StepError("parse step failed")


def trace(text, chained=True):
    try:
        return ("ok", load(text, chained))
    except StepError as exc:
        return ("failed", chain_messages(exc))
```

**Why:** `__context__` is set **automatically** for any exception raised while another is being handled, so the original is never actually lost — `from` only upgrades the relationship to a deliberate `__cause__`, which changes the wording in the traceback and signals intent. The one way to truly drop it is `raise ... from None`, which sets `__suppress_context__` and hides everything underneath.

</details>

In [ ]:
class StepError(Exception):
    ...


def chain_messages(exc):
    ...


def parse(text):
    ...


def load(text, chained):
    ...


def trace(text, chained=True):
    ...

check('Q-356', trace)

### Q-357 - L5 Hard - Decorators

A decorator need not be a function. Write a **class** `Counted` whose `__init__(func)` stores the function, calls `functools.update_wrapper(self, func)`, and sets `self.calls = 0`; its `__call__` increments `self.calls` and delegates.

Then write `measure_calls(values)` that defines a `@Counted`-decorated `negate(n)` **inside** the function, maps it over `values`, and returns the 3-tuple `(negate.__name__, results_list, negate.calls)`.

`measure_calls([1, 2])` returns `('negate', [-1, -2], 2)`.

<details><summary>Hint</summary>

`@Counted` on a `def` binds the name to an *instance*, so calling it goes through `__call__` and the counter is plain instance state. `functools.update_wrapper` is the non-decorator form of `functools.wraps` — it copies metadata onto an object you already have.

</details>

<details><summary>Solution</summary>

```python
import functools


class Counted:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func = func
        self.calls = 0

    def __call__(self, *args, **kwargs):
        self.calls += 1
        return self.func(*args, **kwargs)


def measure_calls(values):
    @Counted
    def negate(n):
        return -n

    results = [negate(v) for v in values]
    return (negate.__name__, results, negate.calls)
```

**Why:** A class-based decorator trades closures for attributes, which makes state like `calls` inspectable and resettable from outside rather than trapped in a cell — the reason `functools.lru_cache` exposes `cache_info()`. The catch is methods: an instance stored on a class is not a descriptor, so `@Counted` on a method loses `self` binding unless you also implement `__get__`.

</details>

In [ ]:
import functools


class Counted:
    ...


def measure_calls(values):
    ...

check('Q-357', measure_calls)

### Q-358 - L5 Hard - Context Managers

Capstone: build `@contextlib.contextmanager` yourself, out of a decorator, a generator and the context-manager protocol. Do not import `contextlib`.

- `_CM(gen)` — a class whose `__enter__` returns `next(self.gen)`, and whose `__exit__` runs `next(self.gen, None)` on the clean path, `self.gen.close()` when the block raised, and returns `False` either way.
- `simple_cm(func)` — a decorator (using `functools.wraps`) returning a factory that calls `func(*args, **kwargs)` and wraps the resulting generator in `_CM`.
- `run_section(name, fail)` — defines a `@simple_cm`-decorated generator `section(events, label)` **inside itself**, which appends `f"enter {label}"`, then in a `try`/`finally` yields `label.upper()` and appends `f"exit {label}"`. Then, with a fresh `events` list: `with section(events, name) as value`, append `f"body {value}"`, raise `ValueError` when `fail` is true; catch that `ValueError` outside the `with` and append `"caught"`. Return `events`.

`run_section("db", False)` returns `['enter db', 'body DB', 'exit db']`; with `fail=True` a fourth entry `'caught'` follows.

<details><summary>Hint</summary>

`gen.close()` throws `GeneratorExit` in at the paused `yield`, which runs the generator's `finally` and then stops it — that is how cleanup happens without you re-raising anything. And `next(gen, None)` is the two-argument form that returns the default instead of raising when the generator is already finished.

</details>

<details><summary>Solution</summary>

```python
import functools


class _CM:
    def __init__(self, gen):
        self.gen = gen

    def __enter__(self):
        return next(self.gen)

    def __exit__(self, exc_type, exc, tb):
        if exc_type is None:
            next(self.gen, None)
        else:
            self.gen.close()
        return False


def simple_cm(func):
    @functools.wraps(func)
    def factory(*args, **kwargs):
        return _CM(func(*args, **kwargs))
    return factory


def run_section(name, fail):
    @simple_cm
    def section(events, label):
        events.append(f"enter {label}")
        try:
            yield label.upper()
        finally:
            events.append(f"exit {label}")

    events = []
    try:
        with section(events, name) as value:
            events.append(f"body {value}")
            if fail:
                raise ValueError("boom")
    except ValueError:
        events.append("caught")
    return events
```

**Why:** Everything `@contextmanager` does is here: the code before the `yield` is `__enter__`, the code after is `__exit__`, and the factory exists so each `with` gets a fresh generator — reusing one would make the manager single-use. The real `contextlib` version uses `gen.throw(exc)` instead of `close()`, so the generator can *catch* the exception and suppress it; `close()` only guarantees `finally` runs, which is why this version can never swallow anything.

</details>

In [ ]:
import functools


class _CM:
    ...


def simple_cm(func):
    ...


def run_section(name, fail):
    ...

check('Q-358', run_section)

---

## Mini-project P-09 - Resilient paginated crawler

Three of this notebook's tools, assembled into one small client.

**1. `retry(times, exceptions)`** — a decorator factory. The wrapper calls the function up to `times` times, swallowing any exception in the `exceptions` tuple, and re-raises the **last** failure with a bare `raise` if every attempt fails. Use `functools.wraps`.

**2. `lru(capacity)`** — a decorator factory caching results in a dict keyed by the positional args. A hit moves the key to most-recently-used (`cache[key] = cache.pop(key)`); a miss that pushes the cache over `capacity` evicts the oldest key (`next(iter(cache))`). Count hits and misses on the wrapper and expose `wrapper.stats()` returning `(hits, misses)`.

**3. `paginate(fetch, start=0)`** — a generator. Starting at `cursor = start`, call `items, cursor = fetch(cursor)` and `yield from items`, stopping when the returned cursor is `None`. It must never build the full list.

**Entry point `crawl(pages, fail_times, capacity, passes)`**:

- `pages` is a list of pages (each a list of items); `fail_times` maps a page index to how many times fetching it fails before succeeding.
- Define `raw_fetch(cursor)`: if that page still owes failures, decrement and raise `ConnectionError`; otherwise return `(list(page), next_cursor)` where `next_cursor` is `cursor + 1` or `None` past the last page. With no pages at all, `raw_fetch(0)` returns `([], None)`.
- Wrap it: `fetch = lru(capacity)(retry(3, (ConnectionError,))(raw_fetch))` — caching **outside** retry, so a cached page is never re-fetched.
- Run `list(paginate(fetch))` `passes` times, keeping the last result, and return `{"items": ..., "hits": h, "misses": m}`.

With two pages and `passes=2`, the second pass is all hits.

<details><summary>Hint</summary>

Copy `fail_times` into a local dict before mutating it, and build `raw_fetch` as a closure over that copy so each `crawl` starts fresh. Put `lru` on the outside: if retry wrapped the cache instead, a cached page would still pay the retry machinery on every pass.

</details>

<details><summary>Solution</summary>

```python
import functools


def retry(times, exceptions=(Exception,)):
    def decorate(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last = None
            for _ in range(times):
                try:
                    return func(*args, **kwargs)
                except exceptions as exc:
                    last = exc
            raise last
        return wrapper
    return decorate


def lru(capacity):
    def decorate(func):
        cache = {}

        @functools.wraps(func)
        def wrapper(*args):
            if args in cache:
                cache[args] = cache.pop(args)
                wrapper.hits += 1
                return cache[args]
            wrapper.misses += 1
            value = func(*args)
            cache[args] = value
            if len(cache) > capacity:
                del cache[next(iter(cache))]
            return value

        wrapper.hits = 0
        wrapper.misses = 0
        wrapper.stats = lambda: (wrapper.hits, wrapper.misses)
        return wrapper
    return decorate


def paginate(fetch, start=0):
    cursor = start
    while cursor is not None:
        items, cursor = fetch(cursor)
        yield from items


def crawl(pages, fail_times, capacity, passes):
    remaining = dict(fail_times)

    def raw_fetch(cursor):
        if remaining.get(cursor, 0) > 0:
            remaining[cursor] -= 1
            raise ConnectionError(f"page {cursor} unavailable")
        page = pages[cursor] if cursor < len(pages) else []
        nxt = cursor + 1 if cursor + 1 < len(pages) else None
        return (list(page), nxt)

    fetch = lru(capacity)(retry(3, (ConnectionError,))(raw_fetch))

    items = []
    for _ in range(passes):
        items = list(paginate(fetch))
    hits, misses = fetch.stats()
    return {"items": items, "hits": hits, "misses": misses}
```

**Why:** Decorator order is the design decision here: `lru(retry(fetch))` means a cached page costs nothing, while `retry(lru(fetch))` would re-enter the retry loop on every call and — worse — could cache a value produced by a partially failed sequence. The generator keeps the crawl streaming, so a million-page source costs one page of memory; and re-raising the *last* exception with a bare `raise` is what preserves the traceback of the attempt that actually gave up.

</details>

In [ ]:
import functools


def retry(times, exceptions=(Exception,)):
    ...


def lru(capacity):
    ...


def paginate(fetch, start=0):
    ...


def crawl(pages, fail_times, capacity, passes):
    ...

check('P-09', crawl)

---

## Your progress

In [ ]:
progress()